<a href="https://colab.research.google.com/github/iDurugkar/practice-2026/blob/main/TorchCode/33_beam_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/33_beam_search.ipynb)

# 🟠 Medium: Beam Search Decoding

Implement **beam search** — the classic decoding algorithm for sequence generation.

### Signature
```python
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token) -> list[int]:
    # log_prob_fn: takes token list, returns (V,) log-probabilities
    # Returns: best sequence (list of ints)
```

### Algorithm
1. Start with `[(0.0, [start_token])]`
2. Each step: expand each beam with top-k next tokens
3. Keep top `beam_width` beams by total log-probability
4. Stop when best beam ends with `eos_token` or `max_len` reached

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.7 MB/s eta 0:00:00


In [2]:
import torch

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
  # maintain beams, expand, prune, return best
  beams = [(0.0, [start_token])]
  for _ in range(max_len):
    new_beams = []
    # generate new candidates:
    for prev_lp, prefix in beams:
      topk = torch.topk(log_prob_fn(prefix), beam_width,)
      tokens, logps = topk.indices, topk.values
      new_beams.extend([(prev_lp + logp, prefix + [token]) for logp, token in zip(logps, tokens)])
    # prune
    pruned_beam = sorted(new_beams, key= lambda x: x[0], reverse=True)[:beam_width]
    beams = pruned_beam
    if pruned_beam[0][1][-1] == eos_token:
      break
  return beams[0][1]



In [4]:
# 🧪 Debug
def simple_fn(tokens):
    lp = torch.full((5,), -10.0)
    lp[min(len(tokens), 4)] = 0.0
    return lp
seq = beam_search(simple_fn, start_token=0, max_len=5, beam_width=2, eos_token=4)
print('Sequence:', seq)

Sequence: [0, tensor(1), tensor(2), tensor(3), tensor(4)]


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('beam_search')


🧪 Testing: Beam Search Decoding (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Returns list starting with start_token (4.8ms)
  ✅ [2/4] Greedy path (beam=1) (0.6ms)
  ✅ [3/4] Beam finds better path than greedy (0.5ms)
  ✅ [4/4] Stops at eos (0.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (6.0ms total)
  Progress saved. Run status() to see your dashboard.

